In [11]:
!pip install numpy gymnasium matplotlib -q


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [17]:
import numpy as np
import random
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt

# Gymnasium Environment Setup
class EquationEnvironment(gym.Env):
    def __init__(self, a_range=(-10, 10), b_range=(-10, 10), x_range=(-10, 10)):
        super(EquationEnvironment, self).__init__()
        self.a_range = a_range
        self.b_range = b_range
        self.x_range = x_range
        
        self.action_space = spaces.Discrete(3)  # Actions: -1, 0, +1
        self.observation_space = spaces.Discrete(21)  # States: x in range -10 to 10
        
        self.agent_guesses = []  # To track agent guesses for visualization
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Randomly generate A and B for the equation
        self.a = random.randint(self.a_range[0], self.a_range[1])
        self.b = random.randint(self.b_range[0], self.b_range[1])
        
        # Ensure A is not 0 to have a valid solution
        while self.a == 0:
            self.a = random.randint(self.a_range[0], self.a_range[1])
        
        # Random initial guess for x
        self.state = random.randint(self.x_range[0], self.x_range[1])
        self.agent_guesses = []  # Reset guesses for plotting
        
        # Map state to observation space index (0-20)
        obs = self.state + 10  # Convert from [-10, 10] to [0, 20]
        return obs, {}

    def step(self, action):
        # Actions: 0=decrease, 1=no change, 2=increase
        if action == 0:
            self.state -= 1
        elif action == 1:
            self.state += 0
        elif action == 2:
            self.state += 1

        # Keep state within bounds
        self.state = np.clip(self.state, self.x_range[0], self.x_range[1])
        
        self.agent_guesses.append(self.state)

        # Reward based on the equation Ax + B
        equation_value = self.a * self.state + self.b
        reward = -(equation_value) ** 2
        
        # Episode ends if close enough to 0 or max steps reached
        done = abs(equation_value) < 0.1 or len(self.agent_guesses) > 50
        
        # Map state to observation space index (0-20)
        obs = self.state + 10  # Convert from [-10, 10] to [0, 20]
        
        return obs, reward, done, False, {}

    def render(self):
        # Enhanced visualization showing expected vs actual values
        x_values = np.linspace(self.x_range[0], self.x_range[1], 100)
        correct_y_values = self.a * x_values + self.b
        
        # Calculate agent's trajectory on the equation
        agent_y_values = [self.a * x + self.b for x in self.agent_guesses]
        
        # Calculate analytical (expected) solution
        optimal_x = -self.b / self.a
        optimal_y = 0  # Since Ax + B = 0

        plt.figure(figsize=(15, 12))
        
        # Plot 1: The equation line with expected vs actual comparison
        plt.subplot(3, 1, 1)
        plt.plot(x_values, correct_y_values, label=f"Equation: y = {self.a}x + {self.b}", color='blue', linewidth=2)
        plt.axhline(y=0, color='black', linestyle='--', alpha=0.5, label='Target (y=0)')
        
        # Plot expected (analytical) solution
        plt.scatter([optimal_x], [optimal_y], color='green', s=150, marker='*', 
                   label=f'Expected Solution: x={optimal_x:.2f}', zorder=5, edgecolors='black', linewidth=2)
        
        if len(self.agent_guesses) > 0:
            # Plot agent's path
            plt.scatter(self.agent_guesses, agent_y_values, color='red', alpha=0.7, s=30, 
                       label="Agent's Path", zorder=3)
            plt.plot(self.agent_guesses, agent_y_values, color='red', alpha=0.5, linestyle='-', linewidth=1)
            
            # Highlight agent's final guess
            final_x = self.agent_guesses[-1]
            final_y = self.a * final_x + self.b
            plt.scatter([final_x], [final_y], color='orange', s=120, marker='X', 
                       label=f'Agent Final: x={final_x}', zorder=4, edgecolors='black', linewidth=1)
            
            # Draw error line between expected and actual
            plt.plot([optimal_x, final_x], [optimal_y, final_y], color='purple', 
                    linestyle=':', linewidth=3, alpha=0.8, label=f'Error: {abs(final_x - optimal_x):.2f}')
        
        plt.xlabel('x')
        plt.ylabel('y = Ax + B')
        plt.title(f"Expected vs Agent Solution: y = {self.a}x + {self.b}")
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        
        # Plot 2: Learning progress showing convergence to expected value
        plt.subplot(3, 1, 2)
        if len(self.agent_guesses) > 1:
            distances = [abs(guess - optimal_x) for guess in self.agent_guesses]
            steps = range(len(distances))
            
            plt.plot(steps, distances, color='orange', linewidth=2, marker='o', markersize=4, label='Distance from Expected')
            plt.axhline(y=0, color='green', linestyle='--', alpha=0.7, label='Perfect Solution')
            
            # Highlight improvement over time
            if len(distances) > 1:
                improvement = distances[0] - distances[-1]
                plt.text(len(distances)/2, max(distances)/2, f'Improvement: {improvement:.2f}', 
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.7),
                        fontsize=10, ha='center')
            
            plt.xlabel('Step')
            plt.ylabel('Distance from Expected x')
            plt.title(f'Convergence Progress (Expected x = {optimal_x:.2f})')
            plt.legend()
            plt.grid(True, alpha=0.3)
        
        # Plot 3: Error analysis - Expected vs Actual comparison
        plt.subplot(3, 1, 3)
        if len(self.agent_guesses) > 0:
            steps = range(len(self.agent_guesses))
            expected_line = [optimal_x] * len(self.agent_guesses)  # Constant expected value
            
            plt.plot(steps, expected_line, color='green', linewidth=3, label=f'Expected x = {optimal_x:.2f}', alpha=0.8)
            plt.plot(steps, self.agent_guesses, color='red', linewidth=2, marker='o', markersize=4, 
                    label='Agent x values', alpha=0.8)
            
            # Fill area between expected and actual to show error
            plt.fill_between(steps, expected_line, self.agent_guesses, 
                           color='yellow', alpha=0.3, label='Error Area')
            
            # Add error metrics text
            final_error = abs(self.agent_guesses[-1] - optimal_x)
            mean_error = np.mean([abs(guess - optimal_x) for guess in self.agent_guesses])
            plt.text(0.02, 0.98, f'Final Error: {final_error:.2f}\nMean Error: {mean_error:.2f}', 
                    transform=plt.gca().transAxes, verticalalignment='top',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))
            
            plt.xlabel('Step')
            plt.ylabel('x value')
            plt.title('Expected vs Agent x Values Over Time')
            plt.legend()
            plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Enhanced equation info with error analysis
        print(f"\n{'='*50}")
        print(f"📊 SOLUTION ANALYSIS")
        print(f"{'='*50}")
        print(f"Equation: {self.a}x + {self.b} = 0")
        print(f"Expected solution: x = {optimal_x:.4f}")
        
        if len(self.agent_guesses) > 0:
            final_guess = self.agent_guesses[-1]
            final_error = abs(final_guess - optimal_x)
            relative_error = (final_error / abs(optimal_x)) * 100 if optimal_x != 0 else float('inf')
            
            print(f"Agent's final guess: x = {final_guess}")
            print(f"Absolute error: {final_error:.4f}")
            print(f"Relative error: {relative_error:.2f}%")
            
            # Error classification
            if final_error < 0.1:
                print(f"✅ Excellent solution! (error < 0.1)")
            elif final_error < 0.5:
                print(f"✓ Good solution (error < 0.5)")
            elif final_error < 1.0:
                print(f"⚠️ Acceptable solution (error < 1.0)")
            else:
                print(f"❌ Poor solution (error ≥ 1.0)")
                
            print(f"Steps taken: {len(self.agent_guesses)}")
            print(f"{'='*50}")

print("✓ Enhanced Gymnasium environment with error visualization created successfully!")

✓ Enhanced Gymnasium environment with error visualization created successfully!


In [18]:
# Q-learning Agent Setup
class QLearningAgent:
    def __init__(self, state_space, action_space, alpha=0.1, gamma=0.9, epsilon=0.1, epsilon_decay=0.995, epsilon_min=0.01):
        self.alpha = alpha  # Learning rate
        self.gamma = gamma  # Discount factor
        self.epsilon = epsilon  # Exploration rate
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.q_table = np.zeros((state_space, action_space))  # Initialize Q-table
        self.action_space = action_space
        self.training_rewards = []
        self.training_episodes_length = []

    def choose_action(self, state):
        """Epsilon-greedy action selection"""
        if random.uniform(0, 1) < self.epsilon:  # Exploration
            return random.choice(range(self.action_space))
        else:  # Exploitation
            return np.argmax(self.q_table[state])

    def update_q_table(self, state, action, reward, next_state, done):
        """Q-learning update rule"""
        if done:
            # If terminal state, no future rewards
            target = reward
        else:
            # Standard Q-learning update
            target = reward + self.gamma * np.max(self.q_table[next_state])
        
        # Update Q-value
        self.q_table[state, action] += self.alpha * (target - self.q_table[state, action])

    def decay_epsilon(self):
        """Decay exploration rate over time"""
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

    def get_policy(self):
        """Return the learned policy (best actions for each state)"""
        return np.argmax(self.q_table, axis=1)

print("✓ Q-learning agent implemented successfully!")

✓ Q-learning agent implemented successfully!


In [19]:
# Run the complete demonstration
# Uncomment the line below to run the full demo
# demonstrate_learning()

# For a quick test, let's create and test the environment with a simple example
print("🔧 Quick Test - Creating environment and agent...")
env = EquationEnvironment()
agent = QLearningAgent(state_space=21, action_space=3)

# Reset environment to get a new equation
state, _ = env.reset()
print(f"Test equation: {env.a}x + {env.b} = 0")
print(f"Analytical solution: x = {-env.b/env.a:.2f}")
print(f"Initial agent state: x = {state - 10}")

# Take a few random actions to demonstrate
print("\\nTaking 5 random actions:")
for step in range(5):
    action = env.action_space.sample()  # Random action
    next_state, reward, done, truncated, _ = env.step(action)
    action_names = ['decrease', 'stay', 'increase']
    current_x = next_state - 10
    equation_value = env.a * current_x + env.b
    print(f"  Step {step+1}: action={action_names[action]}, x={current_x}, Ax+B={equation_value:.2f}, reward={reward:.2f}")
    if done:
        print("  Episode terminated!")
        break

print("\\n✅ Implementation is ready!")
print("\\n📋 To run the full demonstration with training, uncomment 'demonstrate_learning()' above.")
print("\\n🎯 Features implemented:")
print("  • Custom Gymnasium environment for equation solving")
print("  • Q-learning agent with epsilon-greedy exploration")
print("  • Training loop with progress tracking")
print("  • Comprehensive visualization of learning process")
print("  • Analysis of learned Q-table and policy")

🔧 Quick Test - Creating environment and agent...
Test equation: 6x + 3 = 0
Analytical solution: x = -0.50
Initial agent state: x = -7
\nTaking 5 random actions:
  Step 1: action=increase, x=-6, Ax+B=-33.00, reward=-1089.00
  Step 2: action=decrease, x=-7, Ax+B=-39.00, reward=-1521.00
  Step 3: action=increase, x=-6, Ax+B=-33.00, reward=-1089.00
  Step 4: action=increase, x=-5, Ax+B=-27.00, reward=-729.00
  Step 5: action=increase, x=-4, Ax+B=-21.00, reward=-441.00
\n✅ Implementation is ready!
\n📋 To run the full demonstration with training, uncomment 'demonstrate_learning()' above.
\n🎯 Features implemented:
  • Custom Gymnasium environment for equation solving
  • Q-learning agent with epsilon-greedy exploration
  • Training loop with progress tracking
  • Comprehensive visualization of learning process
  • Analysis of learned Q-table and policy


In [20]:
# Training Function with Progress Tracking
def train_agent(agent, env, episodes=1000, verbose=True):
    """Train the Q-learning agent"""
    episode_rewards = []
    episode_lengths = []
    success_rate = []
    
    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        steps = 0
        done = False
        
        while not done:
            action = agent.choose_action(state)
            next_state, reward, done, truncated, _ = env.step(action)
            agent.update_q_table(state, action, reward, next_state, done)
            
            state = next_state
            total_reward += reward
            steps += 1
            
            if steps > 100:  # Prevent infinite loops
                break
        
        # Decay epsilon after each episode
        agent.decay_epsilon()
        
        # Track metrics
        episode_rewards.append(total_reward)
        episode_lengths.append(steps)
        
        # Calculate success rate (last 100 episodes)
        if episode >= 99:
            recent_rewards = episode_rewards[-100:]
            successes = sum(1 for r in recent_rewards if r > -1000)  # Arbitrary success threshold
            success_rate.append(successes / 100)
        
        # Print progress
        if verbose and episode % 100 == 0:
            avg_reward = np.mean(episode_rewards[-100:]) if episode >= 100 else np.mean(episode_rewards)
            avg_length = np.mean(episode_lengths[-100:]) if episode >= 100 else np.mean(episode_lengths)
            current_success = success_rate[-1] if success_rate else 0
            print(f"Episode {episode:4d} | Avg Reward: {avg_reward:8.2f} | Avg Length: {avg_length:5.1f} | Success Rate: {current_success:.2f} | ε: {agent.epsilon:.3f}")
    
    return episode_rewards, episode_lengths, success_rate

# Visualization function for training progress
def plot_training_progress(episode_rewards, episode_lengths, success_rate):
    """Plot training metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: Episode rewards
    axes[0, 0].plot(episode_rewards, alpha=0.3, color='blue')
    if len(episode_rewards) > 100:
        # Moving average
        moving_avg = np.convolve(episode_rewards, np.ones(100)/100, mode='valid')
        axes[0, 0].plot(range(99, len(episode_rewards)), moving_avg, color='red', linewidth=2, label='100-episode moving average')
        axes[0, 0].legend()
    axes[0, 0].set_title('Episode Rewards')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Total Reward')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Episode lengths
    axes[0, 1].plot(episode_lengths, alpha=0.3, color='green')
    if len(episode_lengths) > 100:
        # Moving average
        moving_avg = np.convolve(episode_lengths, np.ones(100)/100, mode='valid')
        axes[0, 1].plot(range(99, len(episode_lengths)), moving_avg, color='red', linewidth=2, label='100-episode moving average')
        axes[0, 1].legend()
    axes[0, 1].set_title('Episode Lengths')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('Steps per Episode')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Success rate
    if success_rate:
        axes[1, 0].plot(range(99, 99 + len(success_rate)), success_rate, color='orange', linewidth=2)
        axes[1, 0].set_title('Success Rate (Last 100 Episodes)')
        axes[1, 0].set_xlabel('Episode')
        axes[1, 0].set_ylabel('Success Rate')
        axes[1, 0].grid(True, alpha=0.3)
        axes[1, 0].set_ylim(0, 1)
    
    # Plot 4: Reward distribution
    axes[1, 1].hist(episode_rewards, bins=50, alpha=0.7, color='purple')
    axes[1, 1].set_title('Reward Distribution')
    axes[1, 1].set_xlabel('Reward')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("✓ Training functions implemented successfully!")

✓ Training functions implemented successfully!


In [21]:
# Main execution and demonstration
def demonstrate_learning():
    """Complete demonstration of the RL equation solver"""
    print("🎯 Reinforcement Learning Equation Solver Demo")
    print("="*50)
    
    # Create environment and agent
    env = EquationEnvironment()
    agent = QLearningAgent(state_space=21, action_space=3, alpha=0.1, gamma=0.9, epsilon=0.3)
    
    print(f"Environment: Solve Ax + B = 0 where A, B ∈ [-10, 10]")
    print(f"State space: x ∈ [-10, 10] (discrete)")
    print(f"Action space: {{decrease, stay, increase}} x")
    print(f"Reward: -(Ax + B)²")
    print()
    
    # Train the agent
    print("🚀 Starting training...")
    episode_rewards, episode_lengths, success_rate = train_agent(agent, env, episodes=1000, verbose=True)
    
    
    print("\\n📊 Training completed! Showing results...")
    
    # Plot training progress
    plot_training_progress(episode_rewards, episode_lengths, success_rate)
    
    # Plot Q-table as heatmap
    plot_q_table_heatmap(agent)
    
    # Test the trained agent on a few episodes
    print("\\n🎮 Testing trained agent on new equations...")
    for test_episode in range(3):
        print(f"\\n--- Test Episode {test_episode + 1} ---")
        state, _ = env.reset()
        print(f"New equation: {env.a}x + {env.b} = 0")
        print(f"Analytical solution: x = {-env.b/env.a:.2f}")
        
        done = False
        steps = 0
        agent.epsilon = 0  # No exploration during testing
        
        while not done and steps < 20:
            action = agent.choose_action(state)
            state, reward, done, truncated, _ = env.step(action)
            steps += 1
            
            if steps <= 5 or done:  # Show first few steps and final result
                action_names = ['decrease', 'stay', 'increase']
                current_x = state - 10  # Convert back to [-10, 10] range
                equation_value = env.a * current_x + env.b
                print(f"  Step {steps}: x={current_x}, Ax+B={equation_value:.2f}, action={action_names[action]}")
        
        print(f"  Final result: x = {state - 10}, steps = {steps}")
        
        # Render the learning trajectory for this episode
        env.render()

# Multi-equation testing function
def test_agent_multiple_equations(agent, env, num_tests=10, max_steps=50, show_individual=True, show_summary=True):
    """
    Test the trained agent on multiple different equations
    
    Parameters:
    - agent: Trained Q-learning agent
    - env: Environment instance
    - num_tests: Number of different equations to test
    - max_steps: Maximum steps per equation
    - show_individual: Whether to show individual equation results
    - show_summary: Whether to show summary statistics
    
    Returns:
    - Dictionary with detailed results and statistics
    """
    print(f"🧪 Testing agent on {num_tests} different equations...")
    print("="*60)
    
    # Store results for analysis
    results = {
        'equations': [],
        'expected_solutions': [],
        'agent_solutions': [],
        'absolute_errors': [],
        'relative_errors': [],
        'steps_taken': [],
        'final_rewards': [],
        'success_status': []
    }
    
    # Temporarily disable exploration for testing
    original_epsilon = agent.epsilon
    agent.epsilon = 0
    
    for test_num in range(num_tests):
        # Reset environment to get new equation
        state, _ = env.reset()
        
        # Store equation parameters
        a, b = env.a, env.b
        expected_x = -b / a
        
        if show_individual:
            print(f"\\nTest {test_num + 1}/{num_tests}: {a}x + {b} = 0")
            print(f"Expected solution: x = {expected_x:.4f}")
        
        # Run agent on this equation
        done = False
        steps = 0
        total_reward = 0
        
        while not done and steps < max_steps:
            action = agent.choose_action(state)
            state, reward, done, truncated, _ = env.step(action)
            total_reward += reward
            steps += 1
        
        # Calculate results
        agent_x = state - 10  # Convert back to [-10, 10] range
        absolute_error = abs(agent_x - expected_x)
        relative_error = (absolute_error / abs(expected_x)) * 100 if expected_x != 0 else float('inf')
        
        # Classify success
        if absolute_error < 0.1:
            success = "Excellent"
        elif absolute_error < 0.5:
            success = "Good"
        elif absolute_error < 1.0:
            success = "Acceptable"
        else:
            success = "Poor"
        
        # Store results
        results['equations'].append((a, b))
        results['expected_solutions'].append(expected_x)
        results['agent_solutions'].append(agent_x)
        results['absolute_errors'].append(absolute_error)
        results['relative_errors'].append(relative_error)
        results['steps_taken'].append(steps)
        results['final_rewards'].append(total_reward)
        results['success_status'].append(success)
        
        if show_individual:
            print(f"Agent solution: x = {agent_x}")
            print(f"Error: {absolute_error:.4f} ({relative_error:.2f}%) - {success}")
            print(f"Steps: {steps}, Final reward: {total_reward:.2f}")
    
    # Restore original epsilon
    agent.epsilon = original_epsilon
    
    if show_summary:
        print_test_summary(results, num_tests)
    
    if num_tests >= 5:  # Only plot if we have enough data
        plot_multiple_test_results(results, num_tests)
    
    return results

def print_test_summary(results, num_tests):
    """Print comprehensive summary of multiple equation tests"""
    print(f"\\n{'='*60}")
    print(f"📈 COMPREHENSIVE TEST SUMMARY ({num_tests} equations)")
    print(f"{'='*60}")
    
    # Calculate statistics
    abs_errors = results['absolute_errors']
    rel_errors = [e for e in results['relative_errors'] if e != float('inf')]
    steps = results['steps_taken']
    
    print(f"\\n📊 ERROR STATISTICS:")
    print(f"Mean absolute error: {np.mean(abs_errors):.4f}")
    print(f"Median absolute error: {np.median(abs_errors):.4f}")
    print(f"Max absolute error: {np.max(abs_errors):.4f}")
    print(f"Min absolute error: {np.min(abs_errors):.4f}")
    print(f"Std absolute error: {np.std(abs_errors):.4f}")
    
    if rel_errors:
        print(f"\\nMean relative error: {np.mean(rel_errors):.2f}%")
        print(f"Median relative error: {np.median(rel_errors):.2f}%")
    
    print(f"\\n⏱️ PERFORMANCE STATISTICS:")
    print(f"Mean steps: {np.mean(steps):.1f}")
    print(f"Median steps: {np.median(steps):.1f}")
    print(f"Max steps: {np.max(steps)}")
    print(f"Min steps: {np.min(steps)}")
    
    # Success rate analysis
    success_counts = {}
    for status in results['success_status']:
        success_counts[status] = success_counts.get(status, 0) + 1
    
    print(f"\\n🎯 SUCCESS RATE BREAKDOWN:")
    for status in ['Excellent', 'Good', 'Acceptable', 'Poor']:
        count = success_counts.get(status, 0)
        percentage = (count / num_tests) * 100
        print(f"{status:12s}: {count:2d}/{num_tests} ({percentage:5.1f}%)")
    
    # Overall success rate (everything except "Poor")
    successful = sum(success_counts.get(status, 0) for status in ['Excellent', 'Good', 'Acceptable'])
    overall_success_rate = (successful / num_tests) * 100
    print(f"\\n🏆 Overall Success Rate: {successful}/{num_tests} ({overall_success_rate:.1f}%)")
    
    # Best and worst cases
    best_idx = np.argmin(abs_errors)
    worst_idx = np.argmax(abs_errors)
    
    print(f"\\n🌟 BEST CASE:")
    eq = results['equations'][best_idx]
    print(f"Equation: {eq[0]}x + {eq[1]} = 0")
    print(f"Expected: {results['expected_solutions'][best_idx]:.4f}, Agent: {results['agent_solutions'][best_idx]}")
    print(f"Error: {abs_errors[best_idx]:.4f}, Steps: {results['steps_taken'][best_idx]}")
    
    print(f"\\n💥 WORST CASE:")
    eq = results['equations'][worst_idx]
    print(f"Equation: {eq[0]}x + {eq[1]} = 0")
    print(f"Expected: {results['expected_solutions'][worst_idx]:.4f}, Agent: {results['agent_solutions'][worst_idx]}")
    print(f"Error: {abs_errors[worst_idx]:.4f}, Steps: {results['steps_taken'][worst_idx]}")
    
    print(f"{'='*60}")

def plot_multiple_test_results(results, num_tests):
    """Visualize results from multiple equation tests"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Expected vs Agent solutions
    axes[0, 0].scatter(results['expected_solutions'], results['agent_solutions'], 
                      alpha=0.7, s=60, c='blue', edgecolors='black')
    # Perfect line
    min_val = min(min(results['expected_solutions']), min(results['agent_solutions']))
    max_val = max(max(results['expected_solutions']), max(results['agent_solutions']))
    axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, alpha=0.8, label='Perfect Solution')
    axes[0, 0].set_xlabel('Expected Solution')
    axes[0, 0].set_ylabel('Agent Solution')
    axes[0, 0].set_title('Expected vs Agent Solutions')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Error distribution
    axes[0, 1].hist(results['absolute_errors'], bins=min(15, num_tests//2), alpha=0.7, color='orange', edgecolor='black')
    axes[0, 1].axvline(np.mean(results['absolute_errors']), color='red', linestyle='--', 
                      label=f'Mean: {np.mean(results["absolute_errors"]):.3f}')
    axes[0, 1].set_xlabel('Absolute Error')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Error Distribution')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Steps taken
    axes[0, 2].hist(results['steps_taken'], bins=min(15, num_tests//2), alpha=0.7, color='green', edgecolor='black')
    axes[0, 2].axvline(np.mean(results['steps_taken']), color='red', linestyle='--', 
                      label=f'Mean: {np.mean(results["steps_taken"]):.1f}')
    axes[0, 2].set_xlabel('Steps Taken')
    axes[0, 2].set_ylabel('Frequency')
    axes[0, 2].set_title('Steps Distribution')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Plot 4: Error vs Steps
    axes[1, 0].scatter(results['steps_taken'], results['absolute_errors'], 
                      alpha=0.7, s=60, c='purple', edgecolors='black')
    axes[1, 0].set_xlabel('Steps Taken')
    axes[1, 0].set_ylabel('Absolute Error')
    axes[1, 0].set_title('Error vs Steps Taken')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 5: Success rate pie chart
    success_counts = {}
    for status in results['success_status']:
        success_counts[status] = success_counts.get(status, 0) + 1
    
    labels = list(success_counts.keys())
    sizes = list(success_counts.values())
    colors = {'Excellent': 'green', 'Good': 'lightgreen', 'Acceptable': 'yellow', 'Poor': 'red'}
    plot_colors = [colors.get(label, 'gray') for label in labels]
    
    axes[1, 1].pie(sizes, labels=labels, colors=plot_colors, autopct='%1.1f%%', startangle=90)
    axes[1, 1].set_title('Success Rate Distribution')
    
    # Plot 6: Error progression by test number
    axes[1, 2].plot(range(1, num_tests + 1), results['absolute_errors'], 
                   marker='o', linewidth=2, markersize=6, alpha=0.8)
    axes[1, 2].axhline(np.mean(results['absolute_errors']), color='red', linestyle='--', 
                      label=f'Mean Error: {np.mean(results["absolute_errors"]):.3f}')
    axes[1, 2].set_xlabel('Test Number')
    axes[1, 2].set_ylabel('Absolute Error')
    axes[1, 2].set_title('Error Progression Across Tests')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Q-table visualization function
def plot_q_table_heatmap(agent):
    """Plot Q-table as a heatmap"""
    plt.figure(figsize=(12, 8))
    
    # Create subplots for each action
    action_names = ['Decrease x', 'Stay', 'Increase x']
    
    # Plot Q-values for each action separately
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    for action in range(3):
        q_values_action = agent.q_table[:, action].reshape(-1, 1)
        
        # Create heatmap
        im = axes[action].imshow(q_values_action.T, cmap='RdYlBu_r', aspect='auto')
        
        # Set labels
        axes[action].set_title(f'Q-values for Action: {action_names[action]}', fontsize=14, fontweight='bold')
        axes[action].set_xlabel('State (x value)', fontsize=12)
        axes[action].set_ylabel('Q-value', fontsize=12)
        
        # Set x-axis ticks to show actual x values (-10 to 10)
        x_labels = [str(i-10) for i in range(21)]
        axes[action].set_xticks(range(21))
        axes[action].set_xticklabels(x_labels, rotation=45)
        axes[action].set_yticks([0])
        axes[action].set_yticklabels(['Q-value'])
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[action])
        cbar.set_label('Q-value', fontsize=10)
        
        # Add text annotations with Q-values
        for i in range(21):
            axes[action].text(i, 0, f'{q_values_action[i, 0]:.1f}', 
                            ha='center', va='center', fontsize=8, 
                            color='white' if abs(q_values_action[i, 0]) > abs(q_values_action.max())/2 else 'black')
    
    plt.tight_layout()
    plt.suptitle('Q-table Heatmap: Q-values for Each State-Action Pair', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    # Also create a combined heatmap showing all actions
    plt.figure(figsize=(14, 8))
    
    # Plot the full Q-table
    im = plt.imshow(agent.q_table.T, cmap='RdYlBu_r', aspect='auto')
    
    plt.title('Complete Q-table: All State-Action Pairs', fontsize=16, fontweight='bold')
    plt.xlabel('State (x value)', fontsize=12)
    plt.ylabel('Action', fontsize=12)
    
    # Set x-axis ticks to show actual x values (-10 to 10)
    x_labels = [str(i-10) for i in range(21)]
    plt.xticks(range(21), x_labels, rotation=45)
    plt.yticks(range(3), action_names)
    
    # Add colorbar
    cbar = plt.colorbar(im)
    cbar.set_label('Q-value', fontsize=12)
    
    # Add text annotations with Q-values
    for i in range(21):
        for j in range(3):
            plt.text(i, j, f'{agent.q_table[i, j]:.1f}', 
                    ha='center', va='center', fontsize=8,
                    color='white' if abs(agent.q_table[i, j]) > abs(agent.q_table.max())/2 else 'black')
    
    plt.tight_layout()
    plt.show()

# Helper function to analyze the learned Q-table
def analyze_q_table(agent, env):
    """Analyze what the agent has learned"""
    print("\\n🧠 Q-table Analysis")
    print("="*30)
    
    # Show policy for each state
    policy = agent.get_policy()
    action_names = ['decrease', 'stay', 'increase']
    
    print("Learned policy:")
    for state_idx in range(21):
        x_value = state_idx - 10  # Convert to actual x value
        best_action = policy[state_idx]
        q_values = agent.q_table[state_idx]
        print(f"  x={x_value:3d}: {action_names[best_action]:8s} (Q-values: {q_values})")

print("✓ Demonstration functions ready!")

✓ Demonstration functions ready!


In [22]:
# Example usage of multi-equation testing
# Uncomment the lines below to run different tests

# demonstrate_learning()

# For quick multi-equation testing after training:
# env = EquationEnvironment()
# agent = QLearningAgent(state_space=21, action_space=3)
# 
# # Train the agent first
# episode_rewards, episode_lengths, success_rate = train_agent(agent, env, episodes=1000, verbose=False)
# 
# # Test on multiple equations
# results = test_agent_multiple_equations(agent, env, num_tests=20, show_individual=True, show_summary=True)

print("📋 To test the agent on multiple equations:")
print("1. First train an agent using train_agent()")
print("2. Then call: test_agent_multiple_equations(agent, env, num_tests=N)")
print("\\nExample usage:")
print("results = test_agent_multiple_equations(agent, env, num_tests=20)")
print("\\nParameters:")
print("- num_tests: Number of different equations to test (default: 10)")
print("- max_steps: Maximum steps per equation (default: 50)")
print("- show_individual: Show each test result (default: True)")
print("- show_summary: Show comprehensive statistics (default: True)")

📋 To test the agent on multiple equations:
1. First train an agent using train_agent()
2. Then call: test_agent_multiple_equations(agent, env, num_tests=N)
\nExample usage:
results = test_agent_multiple_equations(agent, env, num_tests=20)
\nParameters:
- num_tests: Number of different equations to test (default: 10)
- max_steps: Maximum steps per equation (default: 50)
- show_individual: Show each test result (default: True)
- show_summary: Show comprehensive statistics (default: True)
